# Concordances

In [1]:
import polars as pl
import polars.selectors as cs
import polars_corpus as plc
import great_tables as gt

In [2]:
bnc = pl.read_parquet("bnc.parquet")  # .head(5000000)

In [7]:
m = plc.search(bnc, "{tweak}")
m

SearchResults<'{tweak}'; 138 matches>

In [8]:
m.concordance(["token", "pos"]).explode(cs.all()).group_by("token", "pos").len().sort(
    by="len", descending=True
)

token,pos,len
str,str,u32
"""tweaked""","""VERB""",42
"""tweaking""","""VERB""",32
"""tweak""","""VERB""",27
"""tweak""","""SUBST""",22
"""tweaks""","""SUBST""",8
"""tweaks""","""VERB""",2
"""TWEAKS""","""SUBST""",2
"""TWEAK""","""VERB""",1
"""Tweaking""","""VERB""",1


In [10]:
verbs = plc.search(bnc, "{tweak/V}")
verbs

SearchResults<'{tweak/V}'; 106 matches>

In [11]:
verbs.concordance("token", window=1).explode(cs.all()).sort(by="token_right_context")

token_left_context,token,token_right_context
str,str,str
"""presentation""","""tweaking""",""","""
""",""","""tweaking""",""","""
"""not""","""tweaked""",""","""
"""and""","""tweaked""",""","""
"""need""","""tweaking""","""."""
…,…,…
"""of""","""tweaking""","""—"""
"""‘""","""tweak""","""’"""
"""‘""","""tweak""","""’"""


In [12]:
tbl = (
    verbs.concordance("token", window=20)
    .select(cs.all().list.join(" "))
    .with_columns(
        pl.col("token_left_context").str.tail(50),
        pl.col("token_right_context").str.head(50),
    )
    .style
)

tbl.tab_header(title=gt.md("*tweak* in the BNC")).cols_align(
    align="right", columns="token_left_context"
).cols_align(align="center", columns="token").cols_align(
    align="left", columns="token_right_context"
).tab_options(table_font_names=gt.system_fonts("industrial"))

GT(_tbl_data=shape: (106, 3)
┌─────────────────────────────────┬──────────┬─────────────────────────────────┐
│ token_left_context              ┆ token    ┆ token_right_context             │
│ ---                             ┆ ---      ┆ ---                             │
│ str                             ┆ str      ┆ str                             │
╞═════════════════════════════════╪══════════╪═════════════════════════════════╡
│ go out . ’ She bounced to the … ┆ tweak    ┆ for a moment , catching Conroy… │
│ eading the attack by the resty… ┆ tweaked  ┆ Polos is the supercharged G40 … │
│ , Rudd realised that the front… ┆ tweaked  ┆ still further . He formed the … │
│  ) by subjecting his vocal to … ┆ tweaked  ┆ it up to an androgynous 55 rpm… │
│ said Philip . The boy took hol… ┆ tweaking ┆ it . ‘ Did you get that from t… │
│ …                               ┆ …        ┆ …                               │
│  wish to embarrass the Ulster … ┆ tweak    ┆ them back into line . ’ Ulster… │
│ duct , although he cheerfully … ┆ tweaked  ┆ into shape . ‘ There is room f… │
│  bumps ahead and a radio link … ┆ tweak    ┆ the car 's operating systems d… │
│ 's miles more important than t… ┆ tweak    ┆ his ear off if he does n't get… │
│ s it is , maybe he made a font… ┆ tweak    ┆ it . The one I 've seen was th… │
└─────────────────────────────────┴──────────┴─────────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x11672b4d0>, _boxhead=Boxhead([ColInfo(var='token_left_context', type=<ColInfoTypeEnum.default: 1>, column_label='token_left_context', column_align='right', column_width=None), ColInfo(var='token', type=<ColInfoTypeEnum.default: 1>, column_label='token', column_align='center', column_width=None), ColInfo(var='token_right_context', type=<ColInfoTypeEnum.default: 1>, column_label='token_right_context', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x11672b230>, _spanners=Spanners([]), _heading=Heading(title=Md(text='*tweak* in the BNC'), subtitle=None, preheader=None), _stubhead=None, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x11672b8c0>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['Bahnschrift', 'DIN Alternate', 'Franklin Gothic Medium', 'Nimbus Sans Narrow', 'sans-serif-condensed', 'sans-serif', 'Apple Color Emoji', 'Segoe UI Emoji', 'Segoe UI Symbol', 'Noto Color Emoji']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value

In [13]:
tbl = (
    verbs.concordance("token", chunk_tag="sentence_tag")
    .select(cs.all().list.join(" "))
    .select(
        pl.concat_str(
            pl.col("token_left_context"),
            pl.lit(" **")
            + pl.col("token")
            + pl.lit("** ")
            + pl.col("token_right_context"),
        ).alias("sentence")
    )
    .style
)

tbl.tab_header(title=gt.md("*tweak* in the BNC")).cols_align(
    align="left", columns="sentence"
).fmt_markdown(columns="sentence")
#    .tab_options(table_font_names=gt.system_fonts("industrial")) \

tweak in the BNC
sentence
"She bounced to the mirror to powder and tweak for a moment , catching Conroy 's eye and giving him a wink ."
Heading the attack by the restyled and mechanically tweaked Polos is the supercharged G40 packing a 113bhp punch aimed squarely at the Peugeot 205GTi 's jaw .
"Although the P25 was by then quite good , Rudd realised that the front suspension could be tweaked still further ."
With ‘ If I Was Your Girlfriend ’ he created another identity for himself ( ‘ Camille ’ ) by subjecting his vocal to studio wizardry that tweaked it up to an androgynous 55 rpm .
"The boy took hold of the net again , tweaking it ."
"Variants include : Italian tweak — In Italy and Belgium , you can either chose the list of candidates your party has preselected for you , or tick your favourite and tweak his or her name up the preselected list ."
SCHOOLBOYS know how to tweak gadgets .
"Patched together from the remnants of the Docklands ramp which some yonks back had been acquired by Neil , Liverpool 's mini was a pretty unconventional affair — big transistors tweaked with smaller ones , no more then a few feet of flat bottom — and it rode like a ditch ."
"BOTTOM : Fnnnar , beside that , Cameron Black tweaks with the power of a rhino on heat off this fly-off ."


In [ ]:
""